<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/07_lognormal_varying_intercept_slope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 7 — Hierarchical lognormal regression

Notebook 5 let participants differ in both their baseline reaction time and their response to sleep deprivation, using a Gaussian likelihood. Notebook 6 replaced the Gaussian likelihood with a lognormal one, which keeps reaction times positive and makes the daily effect multiplicative; but with only population-level parameters, whole participants sat systematically above or below its predictive bands.

This notebook combines the two: Notebook 5's varying intercepts and slopes, with Notebook 6's lognormal likelihood and log-scale priors. Both pieces are already familiar, so the full model is supplied up front. The task is to **read the combined model**: what each parameter means now that the hierarchy lives on the log scale, how the familiar workflow checks it, and what the fitted hierarchy says about sleep deprivation.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from pymc.stats.log_density import compute_log_density
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

participants = sorted(sleep["Subject"].unique(), key=int)
participant_to_idx = {participant: i for i, participant in enumerate(participants)}
participant_idx = sleep["Subject"].map(participant_to_idx).to_numpy()

assert participant_idx.min() == 0
assert participant_idx.max() == len(participants) - 1
assert np.array_equal(
    np.asarray(participants)[participant_idx],
    sleep["Subject"].to_numpy(),
)

print(f"{len(participants)} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### Plotting helper

The participant plotting helper from the previous notebooks is supplied. It can optionally restrict plots to selected participants using the standard ArviZ `coords` argument.

In [ ]:
LM_VISUALS = {
    "pe_line": {"color": "C1"},
    "ci_band": {"color": "C0"},
    "observed_scatter": {"color": "black", "alpha": 1, "zorder": 3, "s": 10},
}

PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_participants(dt, group, var, coords=None):
    """One panel per participant, optionally restricted with ArviZ coords."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        group: reshape(dt[group].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
        "constant_data": reshape(dt["constant_data"].to_dataset()),
    })

    pc = azp.plot_lm(
        panels,
        x="days",
        y=var,
        y_obs="y",
        group=group,
        plot_dim="day",
        coords=coords,
        ci_prob=(0.50, 0.90),
        ci_kind="hdi",
        point_estimate="mean",
        smooth=False,
        cols=["participant"],
        col_wrap=6,
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={**LM_VISUALS, "xlabel": False, "ylabel": False},
    )

    pc.add_legend("prob", title="HDI")
    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    return pc

## 1. Read the hierarchical lognormal model

### 1.1 The model

The full model is supplied:

$$
y_i \sim \operatorname{LogNormal}(\mu_{y,i}, sd_y)
$$

$$
\mu_{y,i}
=
b_{0,s[i]}
+
b_{1,s[i]}\,\mathrm{days}_i
$$

$$
b_{0,s}
\sim
\operatorname{Normal}(\mu_{b0}, sd_{b0})
$$

$$
b_{1,s}
\sim
\operatorname{Normal}(\mu_{b1}, sd_{b1}).
$$

Here $s[i]$ identifies the participant who produced observation $i$. The mean structure is Notebook 5's and the likelihood is Notebook 6's. Equivalently, $\log y_i \sim \operatorname{Normal}(\mu_{y,i}, sd_y)$: Notebook 5's varying-intercept, varying-slope regression now describes **log** reaction time.

As in Notebook 5, the intercept and slope hierarchies are independent, so the model does not estimate whether participants' baselines and sleep-deprivation effects are correlated.

### 1.2 The complete PyMC implementation

The hierarchy has the same form as in Notebook 5. The priors for the population centers `mu_b0` and `mu_b1` are Notebook 6's priors for `b0` and `b1`, and `sd_y` keeps Notebook 6's residual prior. The between-participant scales `sd_b0` and `sd_b1` have new log-scale priors, examined in Section 2. As in Notebook 6, `mean_rt` is the expected reaction time in milliseconds.

In [ ]:
coords = {
    "obs_id": np.arange(len(sleep)),
    "participant": participants,
}

# Hyperprior constants for the intercept hierarchy (log reaction time).
# The population center reuses Notebook 6's prior for b0.
mu_mu_b0 = 5
sd_mu_b0 = 0.55
sd_sd_b0 = 1 / 6

# Hyperprior constants for the slope hierarchy (change in log reaction time per day).
# The population center reuses Notebook 6's prior for b1.
mu_mu_b1 = 0
sd_mu_b1 = 0.2
sd_sd_b1 = 0.05

# Prior constant for residual variation (log scale), reused from Notebook 6
mu_sd_y = 1 / 3

with pm.Model(coords=coords) as model:
    days = pm.Data("days", sleep["Days"].to_numpy(), dims="obs_id")
    pidx = pm.Data("participant_idx", participant_idx, dims="obs_id")

    # Varying intercepts
    mu_b0 = pm.Normal("mu_b0", mu=mu_mu_b0, sigma=sd_mu_b0)
    sd_b0 = pm.Exponential("sd_b0", scale=sd_sd_b0)
    b0 = pm.Normal("b0", mu=mu_b0, sigma=sd_b0, dims="participant")

    # Varying slopes
    mu_b1 = pm.Normal("mu_b1", mu=mu_mu_b1, sigma=sd_mu_b1)
    sd_b1 = pm.Exponential("sd_b1", scale=sd_sd_b1)
    b1 = pm.Normal("b1", mu=mu_b1, sigma=sd_b1, dims="participant")

    # Likelihood
    sd_y = pm.Exponential("sd_y", scale=mu_sd_y)
    mu_y = pm.Deterministic(
        "mu_y",
        b0[pidx] + b1[pidx] * days,
        dims="obs_id",
    )

    # Expected reaction time in milliseconds
    mean_rt = pm.Deterministic(
        "mean_rt",
        pm.math.exp(mu_y + sd_y**2 / 2),
        dims="obs_id",
    )

    y = pm.LogNormal(
        "y",
        mu=mu_y,
        sigma=sd_y,
        observed=sleep["Reaction"].to_numpy(),
        dims="obs_id",
    )

In [ ]:
pm.model_to_graphviz(model)

### 1.3 Which quantities vary by participant?

Which model variables contain one value for every participant?

- answer here

### 1.4 Which parameter is the population-average daily effect of sleep deprivation, and how does it act on reaction time?

- answer here

### 1.5 Which parameters describe the population distributions of intercepts and slopes?

Which give the centers, and which give the between-participant scales? What does a difference of one `sd_b0` mean for reaction time in milliseconds?

- answer here

### 1.6 What does `sd_y` describe in this model?

`sd_y` keeps Notebook 6's prior, an Exponential distribution with mean 1/3. Would you expect its value to be smaller or larger than in Notebook 6, and why?

- answer here

### 1.7 What does `mean_rt` represent, and how does it differ from `mu_y`?

- answer here

## 2. Check the prior implications

### 2.1 Criteria

The prior predictions should meet the criteria established in Notebook 1: predicted reaction times should not routinely be physically impossible, reaction times near baseline should mostly occupy a broadly plausible range, and the model should allow substantial change across the seven days without routinely generating absurd trajectories.

As in Notebooks 4 and 5, the hierarchy priors should also allow meaningful differences among participants' baselines and sleep-deprivation effects without making enormous differences routine. On the log scale these differences are proportional (Question 1.5).

### 2.2 Draw from the prior and plot the parameter priors.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=500,
        var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "sd_y", "y"],
        random_seed=RANDOM_SEED,
    )

azp.plot_dist(
    prior,
    group="prior",
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "sd_y"],
    ci_prob=0.90,
    ci_kind="hdi",
    point_estimate="mean",
);

### 2.3 Are the hierarchy priors reasonable?

Read `sd_b0` and `sd_b1` multiplicatively. Do they allow meaningful differences between participants without making enormous differences routine? Remember that a difference in daily effect accumulates over the seven days.

- answer here

### 2.4 Plot the prior predictive reaction times.

In [ ]:
plot_participants(prior, "prior_predictive", "y")
plt.show()

### 2.5 Do these prior predictions meet the criteria established in Notebook 1?

Judge support, baseline scale, and changes across days.

- answer here

### 2.6 Why do most panels look similar, but one or two show an extreme spike?

Every participant has their own `b0` and `b1`, drawn from the same population priors.

- answer here

## 3. Fit and diagnose the hierarchy

### 3.1 Sample from the posterior.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        random_seed=RANDOM_SEED,
    )

### 3.2 Check population-level diagnostics.

In [ ]:
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))

azs.summary(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "sd_y"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "sd_y"],
);

### 3.3 Do the population-level parameters meet the diagnostic criteria?

Use the criteria established in Notebook 1: no divergences, R-hat close to 1, adequate bulk and tail ESS, and well-mixed traces.

- answer here

### 3.4 Screen all participants, then inspect a representative subset.

In [ ]:
participant_diagnostics = azs.summary(
    idata,
    var_names=["b0", "b1"],
    kind="diagnostics",
    round_to=2,
)

display(pd.DataFrame(
    {
        "value": [
            participant_diagnostics["r_hat"].max(),
            participant_diagnostics["ess_bulk"].min(),
            participant_diagnostics["ess_tail"].min(),
        ]
    },
    index=["largest R-hat", "smallest bulk ESS", "smallest tail ESS"],
))

diagnostic_participants = [
    participants[0],
    participants[len(participants) // 2],
    participants[-1],
]
diagnostic_coords = {"participant": diagnostic_participants}

azs.summary(
    idata,
    var_names=["b0", "b1"],
    coords=diagnostic_coords,
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["b0", "b1"],
    coords=diagnostic_coords,
);

### 3.5 Do the participant-level parameters meet the same criteria?

- answer here

## 4. Examine the fitted hierarchy

### 4.1 Plot the posterior distribution of the population-average daily effect.

In [ ]:
# answer here

### 4.2 What range of population-average daily effects is credible?

Read the 90% HDI from your plot in Question 4.1 (the summary in Question 3.2 rounds it to two decimals), and express it as a percentage change per day.

- answer here

### 4.3 Plot the posterior distribution of the parameter that controls participant-to-participant variation in daily effects.

In [ ]:
# answer here

### 4.4 What range of slope-variation scales is credible?

Read the 90% HDI from your plot in Question 4.3 (the summary in Question 3.2 rounds it to two decimals), and express it in percentage points per day.

- answer here

### 4.5 Plot the participant-specific daily effects.

In [ ]:
# answer here

### 4.6 Does the fitted model support meaningful heterogeneity in sleep-deprivation effects?

Base the answer on the participant intervals and the posterior for `sd_b1`.

- answer here

### 4.7 How does the fitted `sd_y` compare with Notebook 6's?

Notebook 6's fitted residual scale was about 0.17. Use the posterior summary in Question 3.2.

- answer here

## 5. Predictive consequences

### 5.1 Plot each participant's posterior expected reaction time.

Use `plot_participants` with the variable that gives the expected reaction time in milliseconds.

In [ ]:
# answer here

### 5.2 What uncertainty do these bands represent?

Is it uncertainty about each participant's expected reaction time or uncertainty about future reaction times?

- answer here

### 5.3 Generate and plot posterior predictive reaction times.

Generate replicated values of `y` from the fitted model, add them to `idata`, and compare them with the observations using `plot_participants`.

In [ ]:
# answer here

### 5.4 What is added when we move from `mean_rt` to posterior predictive `y`?

- answer here

### 5.5 Does the model reproduce the participant-level data?

Apply the posterior predictive criteria from Notebook 1: participants' overall levels, changes across days, and residual variation, emphasizing discrepancies that persist across a participant's observations. Also check positivity, and compare with Notebook 6's participant-level check.

- answer here

### 5.6 What would indicate adequate fit in an ECDF check?

The participant panels check conditional trajectories. Pooling all observations asks a different question: does the model reproduce the overall distribution of reaction times?

As in Notebooks 5 and 6, the observed ECDF should be broadly consistent with the posterior-predictive ECDFs across the whole distribution, without a persistent systematic displacement, particularly in the tails.

### 5.7 Plot the observed and posterior-predictive ECDFs.

Use `azp.plot_ppc_dist` with `kind="ecdf"`.

In [ ]:
# answer here

### 5.8 Does the model reproduce the overall distribution, and how does this compare with Notebook 6?

- answer here

## 6. Prior sensitivity

### 6.1 Why check prior sensitivity here, and which priors should be power-scaled?

The prior predictions in Section 2 were only partly plausible, and the between-participant scales `sd_b0` and `sd_b1` are informed by only 18 participants rather than by all 144 observations. Power-scaling sensitivity, introduced in Notebook 1, asks whether slightly strengthening or weakening the priors would move the posterior substantially.

In a hierarchical model, not every prior density is a prior assumption we chose. The distributions $b_{0,s} \sim \operatorname{Normal}(\mu_{b0}, sd_{b0})$ and $b_{1,s} \sim \operatorname{Normal}(\mu_{b1}, sd_{b1})$ are the population model itself: they are how the participants' data inform `mu_b0`, `sd_b0`, `mu_b1`, and `sd_b1`. Power-scaling them would change that structure, and the diagnostic would then flag the hyperparameters for reasons unrelated to the priors we chose. We therefore power-scale only the top-level priors, on `mu_b0`, `sd_b0`, `mu_b1`, `sd_b1`, and `sd_y`, using the `prior_var_names` argument of `azs.psense_summary`.

### 6.2 Prepare the information required for power-scaling.

The bookkeeping code is supplied.

In [ ]:
with model:
    pm.compute_log_likelihood(idata)
    compute_log_density(
        idata,
        model=model,
        kind="prior",
        extend_inferencedata=True,
    )

### 6.3 Calculate prior sensitivity.

Use `azs.psense_summary` for `mu_b0`, `sd_b0`, `mu_b1`, `sd_b1`, and `sd_y`, power-scaling only their priors with `prior_var_names`.

In [ ]:
# answer here

### 6.4 Are the conclusions sensitive to the priors?

Recall that ArviZ uses 0.05 as a screening threshold for prior sensitivity. Does the `diagnosis` column flag any parameter?

- answer here

## 7. Summary

### 7.1 What has this notebook shown?

Summarize what combining the hierarchy with the lognormal likelihood showed about the meaning of the parameters, the priors, the fitted hierarchy, the predictive checks, and prior sensitivity.

- answer here